# Convert outputs to yearly zarr files

In [1]:
import re
import os
import sys

import zarr
import yaml
from glob import glob
from datetime import datetime, timedelta

import numpy as np
import xarray as xr

In [2]:
sys.path.insert(0, os.path.realpath('../libs/'))
import verif_utils as vu

### Get the target data for coord reference

In [3]:
# fn_target = '/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/C404/C404_GP_2020.zarr'
# ds_target = xr.open_zarr(fn_target)

## Gather prognostic outputs

### Combine raw netCDF4 to zarr

In [4]:
# ind_start = 0
# ind_end = 24*366 - 1

In [12]:
source_dir = '/glade/derecho/scratch/ksha/DWC/RAW_OUTPUT/CONUS_GP_B1H/'

fn_all = vu.get_nc_files(source_dir)[1]
fn_all = sorted(fn_all, key=lambda x: int(re.search(r'_(\d+)\.nc$', x).group(1)))[:-1]

In [13]:
fn_all[-1]

'/glade/derecho/scratch/ksha/DWC/RAW_OUTPUT/CONUS_GP_B1H/2023-01-01T00Z/pred_2023-01-01T00Z_17641.nc'

In [8]:
# for fn in fn_all[ind_start:ind_end]:
#     ds = xr.open_dataset(fn)
#     ds_collect.append(ds)

# ds_final = xr.concat(ds_collect, dim='time')

# ds_final = ds_final.rename({'latitude': 'south_north', 'longitude': 'west_east', 'level': 'bottom_top'})
# ds_final['west_east'] = np.arange(336).astype(np.float32)
# ds_final['south_north'] = np.arange(336).astype(np.float32)
# ds_final['bottom_top'] = np.arange(12).astype(np.float32)

# # =================================================== #
# # combine with diag
# # ds_target = ds_target.sel(time=ds_final['time'])
# # ds_final = xr.merge([ds_final, ds_target[['WRF_precip_025', 'WRF_radar_composite_025', 'WRF_OLR', 'WRF_TCC', 'WRF_GLW', 'WRF_SWDOWN', 'WRF_SMOIS', 'WRF_TSLB']]])
# ds_final = ds_final.chunk({'time': 12, 'bottom_top': 12, 'south_north': 336, 'west_east': 336})

# # =================================================== #
# # zarr encodings
# dict_encoding = {}
# varnames = list(ds_final.keys())
# varname_4D = ['WRF_U', 'WRF_V', 'WRF_T', 'WRF_Q_tot_05', 'WRF_P']

# chunk_size_3d = dict(chunks=(12, 336, 336))
# chunk_size_4d = dict(chunks=(12, 12, 336, 336))
# compress = zarr.Blosc(cname='zstd', clevel=1, shuffle=zarr.Blosc.SHUFFLE, blocksize=0)

# for i_var, var in enumerate(varnames):
#     if var in varname_4D:
#         dict_encoding[var] = {'compressor': compress, **chunk_size_4d}
#     else:
#         dict_encoding[var] = {'compressor': compress, **chunk_size_3d}

# save_name = f'/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/opt_single_clean_{ind_start:04d}_{ind_end:04d}_2020-01-01T00Z.zarr'
# # ds_final.to_zarr(save_name, mode='w', consolidated=True, compute=True, encoding=dict_encoding)

### Concat zarr to yearly

In [4]:
def extract_start_index(path):
    # Find numbers like _0000_1000_ and take the first one
    match = re.search(r'_(\d+)_\d+_', path)
    return int(match.group(1)) if match else float('inf')

def year_from_dt64(dt64):
    """Convert numpy.datetime64[ns] to integer year."""
    return dt64.astype("datetime64[Y]").astype(int) + 1970

def first_day_of_year(dt64):
    """Return the datetime64[ns] of the first day of the year for dt64."""
    return dt64.astype("datetime64[Y]").astype("datetime64[ns]")

### B1H

In [14]:
for year in range(2020, 2024):
    base_dir = '/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/B1H/'
    fn_all = sorted(glob(base_dir + '*.zarr'), key=extract_start_index)
    
    ds_collect = []
    flag_add_previous = True
    
    for i_fn, fn in enumerate(fn_all):
        ds = xr.open_zarr(fn)
    
        # Extract first timestamp
        first_day = ds['time'].values[0]
        year_ = year_from_dt64(first_day)
    
        if year_ != year:
            continue  # Skip years not matching target
        
        # If file starts mid-year, add the previous file once
        if (first_day != first_day_of_year(first_day)) and (i_fn > 0) and flag_add_previous and (year != 2020):
            print(fn)
            ds_prev = xr.open_zarr(fn_all[i_fn - 1])
            ds_collect.append(ds_prev)
            flag_add_previous = False
    
        ds_collect.append(ds)
    
    # Combine and trim to exact calendar year
    if not ds_collect:
        raise ValueError(f"No data found for year {year}")
    
    ds_final = xr.concat(ds_collect, dim='time')
    
    ds_final = ds_final.sel(time=slice(f"{year}-01-01T00:00:00", f"{year}-12-31T23:00:00"))
    ds_final = ds_final.chunk({'time': 12, 'south_north': 336, 'west_east': 336, 'bottom_top': 12})
    
    # =================================================== #
    # zarr encodings
    dict_encoding = {}
    varnames = list(ds_final.keys())
    varname_4D = ['WRF_U', 'WRF_V', 'WRF_T', 'WRF_Q_tot_05', 'WRF_P']
    
    chunk_size_3d = dict(chunks=(12, 336, 336))
    chunk_size_4d = dict(chunks=(12, 12, 336, 336))
    compress = zarr.Blosc(cname='zstd', clevel=1, shuffle=zarr.Blosc.SHUFFLE, blocksize=0)
    
    for i_var, var in enumerate(varnames):
        if var in varname_4D:
            dict_encoding[var] = {'compressor': compress, **chunk_size_4d}
        else:
            dict_encoding[var] = {'compressor': compress, **chunk_size_3d}
    
    base_dir = '/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/'
    save_name = base_dir + f'opt_B1H_{year}.zarr'
    ds_final.to_zarr(save_name, mode='w', consolidated=True, compute=True, encoding=dict_encoding)
    print(save_name)

/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/opt_B1H_2020.zarr
/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/B1H/opt_B1H_9000_10000_2020-01-01T00Z.zarr
/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/opt_B1H_2021.zarr
/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/B1H/opt_B1H_18000_19000_2020-01-01T00Z.zarr
/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/opt_B1H_2022.zarr
/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/B1H/opt_B1H_27000_28000_2020-01-01T00Z.zarr
/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/opt_B1H_2023.zarr


In [14]:
for year in range(2024, 2025):
    base_dir = '/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/B1H_p2/'
    fn_all = sorted(glob(base_dir + '*.zarr'), key=extract_start_index)
    
    ds_collect = []
    flag_add_previous = True
    
    for i_fn, fn in enumerate(fn_all):
        ds = xr.open_zarr(fn)
    
        # Extract first timestamp
        first_day = ds['time'].values[0]
        year_ = year_from_dt64(first_day)
    
        if year_ != year:
            continue  # Skip years not matching target
        
        # If file starts mid-year, add the previous file once
        if (first_day != first_day_of_year(first_day)) and (i_fn > 0) and flag_add_previous and (year != 2020):
            print(fn)
            ds_prev = xr.open_zarr(fn_all[i_fn - 1])
            ds_collect.append(ds_prev)
            flag_add_previous = False
    
        ds_collect.append(ds)
    
    # Combine and trim to exact calendar year
    if not ds_collect:
        raise ValueError(f"No data found for year {year}")
    
    ds_final = xr.concat(ds_collect, dim='time')
    
    ds_final = ds_final.sel(time=slice(f"{year}-01-01T00:00:00", f"{year}-12-31T23:00:00"))
    ds_final = ds_final.chunk({'time': 12, 'south_north': 336, 'west_east': 336, 'bottom_top': 12})
    
    # =================================================== #
    # zarr encodings
    dict_encoding = {}
    varnames = list(ds_final.keys())
    varname_4D = ['WRF_U', 'WRF_V', 'WRF_T', 'WRF_Q_tot_05', 'WRF_P']
    
    chunk_size_3d = dict(chunks=(12, 336, 336))
    chunk_size_4d = dict(chunks=(12, 12, 336, 336))
    compress = zarr.Blosc(cname='zstd', clevel=1, shuffle=zarr.Blosc.SHUFFLE, blocksize=0)
    
    for i_var, var in enumerate(varnames):
        if var in varname_4D:
            dict_encoding[var] = {'compressor': compress, **chunk_size_4d}
        else:
            dict_encoding[var] = {'compressor': compress, **chunk_size_3d}
    
    base_dir = '/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/'
    save_name = base_dir + f'opt_B1H_{year}.zarr'
    ds_final.to_zarr(save_name, mode='w', consolidated=True, compute=True, encoding=dict_encoding)
    print(save_name)

/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/B1H_p2/opt_B1H_9000_10000_2020-01-01T00Z.zarr
/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/opt_B1H_2024.zarr


### B3H

In [18]:
for year in range(2020, 2024):
    base_dir = '/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/B3H/'
    fn_all = sorted(glob(base_dir + '*.zarr'), key=extract_start_index)
    
    ds_collect = []
    flag_add_previous = True
    
    for i_fn, fn in enumerate(fn_all):
        ds = xr.open_zarr(fn)
    
        # Extract first timestamp
        first_day = ds['time'].values[0]
        year_ = year_from_dt64(first_day)
    
        if year_ != year:
            continue  # Skip years not matching target
        
        # If file starts mid-year, add the previous file once
        if (first_day != first_day_of_year(first_day)) and (i_fn > 0) and flag_add_previous and (year != 2020):
            print(fn)
            ds_prev = xr.open_zarr(fn_all[i_fn - 1])
            ds_collect.append(ds_prev)
            flag_add_previous = False
    
        ds_collect.append(ds)
    
    # Combine and trim to exact calendar year
    if not ds_collect:
        raise ValueError(f"No data found for year {year}")
    
    ds_final = xr.concat(ds_collect, dim='time')
    
    ds_final = ds_final.sel(time=slice(f"{year}-01-01T00:00:00", f"{year}-12-31T23:00:00"))
    ds_final = ds_final.chunk({'time': 12, 'south_north': 336, 'west_east': 336, 'bottom_top': 12})
    
    # =================================================== #
    # zarr encodings
    dict_encoding = {}
    varnames = list(ds_final.keys())
    varname_4D = ['WRF_U', 'WRF_V', 'WRF_T', 'WRF_Q_tot_05', 'WRF_P']
    
    chunk_size_3d = dict(chunks=(12, 336, 336))
    chunk_size_4d = dict(chunks=(12, 12, 336, 336))
    compress = zarr.Blosc(cname='zstd', clevel=1, shuffle=zarr.Blosc.SHUFFLE, blocksize=0)
    
    for i_var, var in enumerate(varnames):
        if var in varname_4D:
            dict_encoding[var] = {'compressor': compress, **chunk_size_4d}
        else:
            dict_encoding[var] = {'compressor': compress, **chunk_size_3d}
    
    base_dir = '/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/'
    save_name = base_dir + f'opt_B3H_{year}.zarr'
    ds_final.to_zarr(save_name, mode='w', consolidated=True, compute=True, encoding=dict_encoding)
    print(save_name)

/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/opt_B3H_2020.zarr
/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/B3H/opt_B3H_9000_10000_2020-01-01T00Z.zarr
/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/opt_B3H_2021.zarr
/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/B3H/opt_B3H_18000_19000_2020-01-01T00Z.zarr
/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/opt_B3H_2022.zarr
/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/B3H/opt_B3H_27000_28000_2020-01-01T00Z.zarr
/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/opt_B3H_2023.zarr


In [ ]:
for year in range(2024, 2025):
    base_dir = '/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/B3H_p2/'
    fn_all = sorted(glob(base_dir + '*.zarr'), key=extract_start_index)
    
    ds_collect = []
    flag_add_previous = True
    
    for i_fn, fn in enumerate(fn_all):
        ds = xr.open_zarr(fn)
    
        # Extract first timestamp
        first_day = ds['time'].values[0]
        year_ = year_from_dt64(first_day)
    
        if year_ != year:
            continue  # Skip years not matching target
        
        # If file starts mid-year, add the previous file once
        if (first_day != first_day_of_year(first_day)) and (i_fn > 0) and flag_add_previous and (year != 2020):
            print(fn)
            ds_prev = xr.open_zarr(fn_all[i_fn - 1])
            ds_collect.append(ds_prev)
            flag_add_previous = False
    
        ds_collect.append(ds)
    
    # Combine and trim to exact calendar year
    if not ds_collect:
        raise ValueError(f"No data found for year {year}")
    
    ds_final = xr.concat(ds_collect, dim='time')
    
    ds_final = ds_final.sel(time=slice(f"{year}-01-01T00:00:00", f"{year}-12-31T23:00:00"))
    ds_final = ds_final.chunk({'time': 12, 'south_north': 336, 'west_east': 336, 'bottom_top': 12})
    
    # =================================================== #
    # zarr encodings
    dict_encoding = {}
    varnames = list(ds_final.keys())
    varname_4D = ['WRF_U', 'WRF_V', 'WRF_T', 'WRF_Q_tot_05', 'WRF_P']
    
    chunk_size_3d = dict(chunks=(12, 336, 336))
    chunk_size_4d = dict(chunks=(12, 12, 336, 336))
    compress = zarr.Blosc(cname='zstd', clevel=1, shuffle=zarr.Blosc.SHUFFLE, blocksize=0)
    
    for i_var, var in enumerate(varnames):
        if var in varname_4D:
            dict_encoding[var] = {'compressor': compress, **chunk_size_4d}
        else:
            dict_encoding[var] = {'compressor': compress, **chunk_size_3d}
    
    base_dir = '/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/'
    save_name = base_dir + f'opt_B3H_{year}.zarr'
    ds_final.to_zarr(save_name, mode='w', consolidated=True, compute=True, encoding=dict_encoding)
    print(save_name)

### B6H

In [19]:
for year in range(2020, 2024):
    base_dir = '/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/B6H/'
    fn_all = sorted(glob(base_dir + '*.zarr'), key=extract_start_index)
    
    ds_collect = []
    flag_add_previous = True
    
    for i_fn, fn in enumerate(fn_all):
        ds = xr.open_zarr(fn)
    
        # Extract first timestamp
        first_day = ds['time'].values[0]
        year_ = year_from_dt64(first_day)
    
        if year_ != year:
            continue  # Skip years not matching target
        
        # If file starts mid-year, add the previous file once
        if (first_day != first_day_of_year(first_day)) and (i_fn > 0) and flag_add_previous and (year != 2020):
            print(fn)
            ds_prev = xr.open_zarr(fn_all[i_fn - 1])
            ds_collect.append(ds_prev)
            flag_add_previous = False
    
        ds_collect.append(ds)
    
    # Combine and trim to exact calendar year
    if not ds_collect:
        raise ValueError(f"No data found for year {year}")
    
    ds_final = xr.concat(ds_collect, dim='time')
    
    ds_final = ds_final.sel(time=slice(f"{year}-01-01T00:00:00", f"{year}-12-31T23:00:00"))
    ds_final = ds_final.chunk({'time': 12, 'south_north': 336, 'west_east': 336, 'bottom_top': 12})
    
    # =================================================== #
    # zarr encodings
    dict_encoding = {}
    varnames = list(ds_final.keys())
    varname_4D = ['WRF_U', 'WRF_V', 'WRF_T', 'WRF_Q_tot_05', 'WRF_P']
    
    chunk_size_3d = dict(chunks=(12, 336, 336))
    chunk_size_4d = dict(chunks=(12, 12, 336, 336))
    compress = zarr.Blosc(cname='zstd', clevel=1, shuffle=zarr.Blosc.SHUFFLE, blocksize=0)
    
    for i_var, var in enumerate(varnames):
        if var in varname_4D:
            dict_encoding[var] = {'compressor': compress, **chunk_size_4d}
        else:
            dict_encoding[var] = {'compressor': compress, **chunk_size_3d}
    
    base_dir = '/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/'
    save_name = base_dir + f'opt_B6H_{year}.zarr'
    ds_final.to_zarr(save_name, mode='w', consolidated=True, compute=True, encoding=dict_encoding)
    print(save_name)

/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/opt_B6H_2020.zarr
/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/B6H/opt_B6H_9000_10000_2020-01-01T00Z.zarr
/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/opt_B6H_2021.zarr
/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/B6H/opt_B6H_18000_19000_2020-01-01T00Z.zarr
/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/opt_B6H_2022.zarr
/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/B6H/opt_B6H_27000_28000_2020-01-01T00Z.zarr
/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/opt_B6H_2023.zarr


In [ ]:
for year in range(2024, 2025):
    base_dir = '/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/B6H_p2/'
    fn_all = sorted(glob(base_dir + '*.zarr'), key=extract_start_index)
    
    ds_collect = []
    flag_add_previous = True
    
    for i_fn, fn in enumerate(fn_all):
        ds = xr.open_zarr(fn)
    
        # Extract first timestamp
        first_day = ds['time'].values[0]
        year_ = year_from_dt64(first_day)
    
        if year_ != year:
            continue  # Skip years not matching target
        
        # If file starts mid-year, add the previous file once
        if (first_day != first_day_of_year(first_day)) and (i_fn > 0) and flag_add_previous and (year != 2020):
            print(fn)
            ds_prev = xr.open_zarr(fn_all[i_fn - 1])
            ds_collect.append(ds_prev)
            flag_add_previous = False
    
        ds_collect.append(ds)
    
    # Combine and trim to exact calendar year
    if not ds_collect:
        raise ValueError(f"No data found for year {year}")
    
    ds_final = xr.concat(ds_collect, dim='time')
    
    ds_final = ds_final.sel(time=slice(f"{year}-01-01T00:00:00", f"{year}-12-31T23:00:00"))
    ds_final = ds_final.chunk({'time': 12, 'south_north': 336, 'west_east': 336, 'bottom_top': 12})
    
    # =================================================== #
    # zarr encodings
    dict_encoding = {}
    varnames = list(ds_final.keys())
    varname_4D = ['WRF_U', 'WRF_V', 'WRF_T', 'WRF_Q_tot_05', 'WRF_P']
    
    chunk_size_3d = dict(chunks=(12, 336, 336))
    chunk_size_4d = dict(chunks=(12, 12, 336, 336))
    compress = zarr.Blosc(cname='zstd', clevel=1, shuffle=zarr.Blosc.SHUFFLE, blocksize=0)
    
    for i_var, var in enumerate(varnames):
        if var in varname_4D:
            dict_encoding[var] = {'compressor': compress, **chunk_size_4d}
        else:
            dict_encoding[var] = {'compressor': compress, **chunk_size_3d}
    
    base_dir = '/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/'
    save_name = base_dir + f'opt_B6H_{year}.zarr'
    ds_final.to_zarr(save_name, mode='w', consolidated=True, compute=True, encoding=dict_encoding)
    print(save_name)

### GDAS

In [6]:
for year in range(2020, 2024):
    base_dir = '/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/GDAS/'
    fn_all = sorted(glob(base_dir + '*.zarr'), key=extract_start_index)
    
    ds_collect = []
    flag_add_previous = True
    
    for i_fn, fn in enumerate(fn_all):
        ds = xr.open_zarr(fn)
        
        # Extract first timestamp
        first_day = ds['time'].values[0]
        year_ = year_from_dt64(first_day)
    
        if year_ != year:
            continue  # Skip years not matching target
        
        # If file starts mid-year, add the previous file once
        if (first_day != first_day_of_year(first_day)) and (i_fn > 0) and flag_add_previous and (year != 2020):
            print(fn)
            ds_prev = xr.open_zarr(fn_all[i_fn - 1])
            ds_collect.append(ds_prev)
            flag_add_previous = False
    
        ds_collect.append(ds)
    
    # Combine and trim to exact calendar year
    if not ds_collect:
        raise ValueError(f"No data found for year {year}")
    
    ds_final = xr.concat(ds_collect, dim='time')
    
    ds_final = ds_final.sel(time=slice(f"{year}-01-01T00:00:00", f"{year}-12-31T23:00:00"))
    ds_final = ds_final.chunk({'time': 12, 'south_north': 336, 'west_east': 336, 'bottom_top': 12})
    
    # =================================================== #
    # zarr encodings
    dict_encoding = {}
    varnames = list(ds_final.keys())
    varname_4D = ['WRF_U', 'WRF_V', 'WRF_T', 'WRF_Q_tot_05', 'WRF_P']
    
    chunk_size_3d = dict(chunks=(12, 336, 336))
    chunk_size_4d = dict(chunks=(12, 12, 336, 336))
    compress = zarr.Blosc(cname='zstd', clevel=1, shuffle=zarr.Blosc.SHUFFLE, blocksize=0)
    
    for i_var, var in enumerate(varnames):
        if var in varname_4D:
            dict_encoding[var] = {'compressor': compress, **chunk_size_4d}
        else:
            dict_encoding[var] = {'compressor': compress, **chunk_size_3d}
    
    base_dir = '/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/'
    save_name = base_dir + f'opt_GDAS_{year}.zarr'
    ds_final.to_zarr(save_name, mode='w', consolidated=True, compute=True, encoding=dict_encoding)
    print(save_name)

/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/opt_GDAS_2020.zarr
/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/GDAS/opt_GDAS_9000_10000_2020-01-01T00Z.zarr
/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/opt_GDAS_2021.zarr
/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/GDAS/opt_GDAS_18000_19000_2020-01-01T00Z.zarr
/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/opt_GDAS_2022.zarr
/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/GDAS/opt_GDAS_27000_28000_2020-01-01T00Z.zarr
/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/opt_GDAS_2023.zarr


In [5]:
for year in range(2024, 2025):
    base_dir = '/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/GDAS_p2/'
    fn_all = sorted(glob(base_dir + '*.zarr'), key=extract_start_index)
    
    ds_collect = []
    flag_add_previous = True
    
    for i_fn, fn in enumerate(fn_all):
        ds = xr.open_zarr(fn)
        
        # Extract first timestamp
        first_day = ds['time'].values[0]
        year_ = year_from_dt64(first_day)
    
        if year_ != year:
            continue  # Skip years not matching target
        
        # If file starts mid-year, add the previous file once
        if (first_day != first_day_of_year(first_day)) and (i_fn > 0) and flag_add_previous and (year != 2020):
            print(fn)
            ds_prev = xr.open_zarr(fn_all[i_fn - 1])
            ds_collect.append(ds_prev)
            flag_add_previous = False
    
        ds_collect.append(ds)
    
    # Combine and trim to exact calendar year
    if not ds_collect:
        raise ValueError(f"No data found for year {year}")
    
    ds_final = xr.concat(ds_collect, dim='time')
    
    ds_final = ds_final.sel(time=slice(f"{year}-01-01T00:00:00", f"{year}-12-31T23:00:00"))
    ds_final = ds_final.chunk({'time': 12, 'south_north': 336, 'west_east': 336, 'bottom_top': 12})
    
    # =================================================== #
    # zarr encodings
    dict_encoding = {}
    varnames = list(ds_final.keys())
    varname_4D = ['WRF_U', 'WRF_V', 'WRF_T', 'WRF_Q_tot_05', 'WRF_P']
    
    chunk_size_3d = dict(chunks=(12, 336, 336))
    chunk_size_4d = dict(chunks=(12, 12, 336, 336))
    compress = zarr.Blosc(cname='zstd', clevel=1, shuffle=zarr.Blosc.SHUFFLE, blocksize=0)
    
    for i_var, var in enumerate(varnames):
        if var in varname_4D:
            dict_encoding[var] = {'compressor': compress, **chunk_size_4d}
        else:
            dict_encoding[var] = {'compressor': compress, **chunk_size_3d}
    
    base_dir = '/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/'
    save_name = base_dir + f'opt_GDAS_{year}.zarr'
    ds_final.to_zarr(save_name, mode='w', consolidated=True, compute=True, encoding=dict_encoding)
    print(save_name)

/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/GDAS_p2/opt_GDAS_9000_10000_2023-01-01T00Z.zarr
/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/opt_GDAS_2024.zarr


### Check NaNs

In [29]:
def check_nans_ds(ds):
    nan_vars = []
    for var in ds.data_vars:
        # Check if there are any NaNs in the variable
        if ds[var].isnull().any():
            nan_vars.append(var)
    return nan_vars

In [30]:
# ds_temp = ds_final
# nan_vars = check_nans_ds(ds_temp)

# if nan_vars:
#     print('Dataset contains NaNs in the following variables:')
#     for var in nan_vars:
#         print(f"- {nan_vars}")
#     print(f"File: {fn}")
# else:
#     print('Dataset does not contain NaNs')

### Add 2021 for diag model

In [4]:
fn_target = '/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/C404/C404_GP_2021.zarr'
ds_target = xr.open_zarr(fn_target)
ds_save = ds_target.isel(time=slice(120))
save_name = '/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/opt_m3_clean_2021-01-01T00Z.zarr'
# ds_save.to_zarr(save_name, mode='w', consolidated=True, compute=True)

### To netCDF

In [3]:
ds_geo = xr.open_zarr('/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/static/C404_GP_static.zarr')
XLONG = ds_geo['XLONG'].values
XLAT = ds_geo['XLAT'].values

**Merge with diag**

In [4]:
# save_name = '/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/diag_outputs/diag_2020-01-01T00Z.zarr'
# ds_diag = xr.open_zarr(save_name)
# ds_diag = ds_diag.drop_vars(['forecast_hour'])

In [5]:
# ds_final = ds_final.isel(time=slice(1, None))
# ds_final = xr.merge([ds_final, ds_diag])
# ds_final['WRF_precip'] = ds_final['WRF_precip_025']**4
# ds_final['WRF_radar_composite'] = ds_final['WRF_radar_composite_025']**4
# ds_final = ds_final.drop_vars(['WRF_precip_025', 'WRF_radar_composite_025'])

**CF**

In [9]:
# Load dataset
save_name = '/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/opt_L11_2020-01-01T00Z.zarr'
ds_final = xr.open_zarr(save_name, chunks={})  # lazy load with Dask

# Square variables and drop originals
ds_final = ds_final.assign(
    WRF_PWAT=ds_final['WRF_PWAT_05'] ** 2,
    WRF_Q_tot=ds_final['WRF_Q_tot_05'] ** 2
).drop_vars(['WRF_PWAT_05', 'WRF_Q_tot_05'])

ds_final['WRF_precip'] = ds_final['WRF_precip_025']**4
ds_final['WRF_radar_composite'] = ds_final['WRF_radar_composite_025']**4
ds_final = ds_final.drop_vars(['WRF_precip_025', 'WRF_radar_composite_025'])

# Rename dimensions
ds_final = ds_final.rename({
    'bottom_top': 'level',
    'south_north': 'latitude',
    'west_east': 'longitude'
})

# Subset time
ds_final = ds_final.isel(time=slice(None, -1))

# Assign level coordinate
ds_final = ds_final.assign_coords(level=[0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 21, 24, 30, 36, 42])

# Drop old lat/lon and assign new ones
ds_final = ds_final.drop_vars(['latitude', 'longitude'], errors='ignore').assign_coords(
    XLAT=(('latitude', 'longitude'), XLAT),
    XLONG=(('latitude', 'longitude'), XLONG)
)

# Add global attribute
ds_final.attrs['Conventions'] = 'CF-1.11'

# Decode time
ds_final['time'] = xr.decode_cf(ds_final[['time']]).time

# Replace indexing dims with coordinate dims and standard names
if 'XLAT' in ds_final and 'XLONG' in ds_final:
    ds_final = ds_final.set_coords(['XLAT', 'XLONG'])
    ds_final['XLAT'] = ds_final['XLAT'].rename({'latitude': 'south_north', 'longitude': 'west_east'})
    ds_final['XLONG'] = ds_final['XLONG'].rename({'latitude': 'south_north', 'longitude': 'west_east'})

    ds_final['XLAT'].attrs.update({
        "standard_name": "latitude",
        "units": "degrees_north",
        "long_name": "latitude"
    })
    ds_final['XLONG'].attrs.update({
        "standard_name": "longitude",
        "units": "degrees_east",
        "long_name": "longitude"
    })

# Replace lat/lon dims with south_north/west_east
dim_map_3d = ('time', 'level', 'south_north', 'west_east')
dim_map_2d = ('time', 'south_north', 'west_east')

for var in ds_final.data_vars:
    dims = ds_final[var].dims
    dim_rename = {}
    if 'latitude' in dims:
        dim_rename['latitude'] = 'south_north'
    if 'longitude' in dims:
        dim_rename['longitude'] = 'west_east'
    
    if dim_rename:
        ds_final[var] = ds_final[var].rename(dim_rename)

In [10]:
output_name = '/glade/derecho/scratch/ksha/CONUS_GP_20250520_new/step12_2020010100Z_create_date_20250715.nc'
ds_final.to_netcdf(output_name, format='NETCDF4_CLASSIC')

In [11]:
xr.open_dataset('/glade/derecho/scratch/ksha/CONUS_GP_20250520_new/step12_2020010100Z_create_date_20250715.nc')

<xarray.Dataset>
Dimensions:              (time: 8783, south_north: 336, west_east: 336,
                          level: 15)
Coordinates:
  * level                (level) int32 0 2 4 6 8 10 12 14 16 18 21 24 30 36 42
  * time                 (time) datetime64[ns] 2020-01-01T01:00:00 ... 2020-1...
    XLAT                 (south_north, west_east) float32 ...
    XLONG                (south_north, west_east) float32 ...
Dimensions without coordinates: south_north, west_east
Data variables: (12/21)
    WRF_GLW              (time, south_north, west_east) float32 ...
    WRF_IVT_U            (time, south_north, west_east) float32 ...
    WRF_IVT_V            (time, south_north, west_east) float32 ...
    WRF_MSLP             (time, south_north, west_east) float32 ...
    WRF_OLR              (time, south_north, west_east) float32 ...
    WRF_P                (time, level, south_north, west_east) float32 ...
    ...                   ...
    WRF_V10              (time, south_north, west_east) float32 ...
    forecast_hour        (time) int32 ...
    WRF_PWAT             (time, south_north, west_east) float32 ...
    WRF_Q_tot            (time, level, south_north, west_east) float32 ...
    WRF_precip           (time, south_north, west_east) float32 ...
    WRF_radar_composite  (time, south_north, west_east) float32 ...
Attributes:
    Conventions:  CF-1.11

In [29]:
# save_name = '/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/opt_2020-01-01T00Z.zarr'
# ds_final = xr.open_zarr(save_name)

# ds_final['WRF_PWAT'] = ds_final['WRF_PWAT_05']**2
# ds_final['WRF_Q_tot'] = ds_final['WRF_Q_tot_05']**2
# ds_final = ds_final.drop_vars(['WRF_PWAT_05', 'WRF_Q_tot_05'])
# ds_final = ds_final.rename({'bottom_top': 'level', 'south_north': 'latitude', 'west_east': 'longitude'})

# ds_final = ds_final.isel(time=slice(None, -1))
# ds_final['level'] = [0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 21, 24, 30, 36, 42]


# ds_final = ds_final.drop_vars(['longitude', 'latitude'])
# ds_final = ds_final.assign_coords(
#     XLAT=(('latitude', 'longitude'), XLAT),
#     XLONG=(('latitude', 'longitude'), XLONG)
# )

# ds_final.attrs['Conventions'] = 'CF-1.11'

# ds_final['time'] = xr.decode_cf(ds_final[['time']]).time

# # Replace indexing dims with coordinate dims
# if 'XLAT' in ds_final and 'XLONG' in ds_final:
#     ds_final = ds_final.set_coords(['XLAT', 'XLONG'])
#     ds_final['XLAT'] = ds_final['XLAT'].rename({'latitude': 'south_north', 'longitude': 'west_east'})
#     ds_final['XLONG'] = ds_final['XLONG'].rename({'latitude': 'south_north', 'longitude': 'west_east'})
    

# # Set CF-compliant attributes for lat/lon
# ds_final['XLAT'].attrs = {
#     "standard_name": "latitude",
#     "units": "degrees_north",
#     "long_name": "latitude"
# }
# ds_final['XLONG'].attrs = {
#     "standard_name": "longitude",
#     "units": "degrees_east",
#     "long_name": "longitude"
# }

# # Replace ('latitude', 'longitude') with ('south_north', 'west_east')
# for var in ds_final.data_vars:
#     dims = ds_final[var].dims
#     # 3D fields
#     if ('latitude' in dims or 'longitude' in dims) and 'level' in dims:
#         data = ds_final[var].data
#         ds_final = ds_final.drop_vars(var)
#         ds_final[var] = (('time', 'level', 'south_north', 'west_east'), data)
#     # 2D fields
#     elif 'latitude' in dims or 'longitude' in dims:
#         data = ds_final[var].data
#         ds_final = ds_final.drop_vars(var)
#         ds_final[var] = (('time', 'south_north', 'west_east'), data)

In [18]:
# time_encoding = {
#     "units": "hours since 1900-01-01 00:00:00",
#     "calendar": "gregorian"
# }

# ds_final.to_netcdf(
#     output_name,  
#     format='NETCDF4', 
#     encoding={'time': time_encoding}, 
#     mode='w'
# )

### To netCDF target

In [7]:
# Load dataset lazily
ds_C404 = xr.open_zarr('/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/C404/C404_GP_2020.zarr', chunks={})

# Drop unused variables
drop_vars = [
    'WRF_precip_025', 'WRF_radar_composite_025',
    'WRF_PWAT_05', 'WRF_Q_tot_05'
]
ds_C404 = ds_C404.drop_vars(drop_vars, errors='ignore')

# Rename dimensions
ds_C404 = ds_C404.rename({
    'bottom_top': 'level',
    'south_north': 'latitude',
    'west_east': 'longitude'
})

# Assign level coordinate
ds_C404 = ds_C404.assign_coords(level=[0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 21, 24, 30, 36, 42])

# Drop old lat/lon coords if present, then assign new ones
ds_C404 = ds_C404.drop_vars(['latitude', 'longitude'], errors='ignore').assign_coords(
    XLAT=(('latitude', 'longitude'), XLAT),
    XLONG=(('latitude', 'longitude'), XLONG)
)

# Set global attribute
ds_C404.attrs['Conventions'] = 'CF-1.11'

# Decode CF time
ds_C404['time'] = xr.decode_cf(ds_C404[['time']]).time

# Set coordinate variables and rename dims
if 'XLAT' in ds_C404 and 'XLONG' in ds_C404:
    ds_C404 = ds_C404.set_coords(['XLAT', 'XLONG'])

    # Rename only the coordinate dims, not data
    ds_C404['XLAT'] = ds_C404['XLAT'].rename({'latitude': 'south_north', 'longitude': 'west_east'})
    ds_C404['XLONG'] = ds_C404['XLONG'].rename({'latitude': 'south_north', 'longitude': 'west_east'})

    # Set CF metadata
    ds_C404['XLAT'].attrs.update({
        "standard_name": "latitude",
        "units": "degrees_north",
        "long_name": "latitude"
    })
    ds_C404['XLONG'].attrs.update({
        "standard_name": "longitude",
        "units": "degrees_east",
        "long_name": "longitude"
    })

# Rename lat/lon dims in data variables
for var in ds_C404.data_vars:
    dims = ds_C404[var].dims
    rename_dims = {}
    if 'latitude' in dims:
        rename_dims['latitude'] = 'south_north'
    if 'longitude' in dims:
        rename_dims['longitude'] = 'west_east'
    if rename_dims:
        ds_C404[var] = ds_C404[var].rename(rename_dims)

In [8]:
output_name = '/glade/derecho/scratch/ksha/CONUS_GP_20250520_new/target_2020010100Z_create_date_20250709.nc'
ds_C404.to_netcdf(output_name, format='NETCDF4_CLASSIC')

In [ ]:
/glade/derecho/scratch/ksha/CONUS_GP_20250520_new/opt_2020010100Z_create_date_20250709.nc
/glade/derecho/scratch/ksha/CONUS_GP_20250520_new/target_2020010100Z_create_date_20250709.nc

In [9]:
ds_C404

<xarray.Dataset>
Dimensions:              (time: 8784, south_north: 336, west_east: 336,
                          level: 15, pressure_approx: 15)
Coordinates:
  * level                (level) int64 0 2 4 6 8 10 12 14 16 18 21 24 30 36 42
  * pressure_approx      (pressure_approx) float32 1e+03 960.0 ... 180.0 100.0
  * time                 (time) datetime64[ns] 2020-01-01 ... 2020-12-31T23:0...
    XLAT                 (south_north, west_east) float32 27.87 27.87 ... 39.64
    XLONG                (south_north, west_east) float32 -102.5 ... -87.34
Dimensions without coordinates: south_north, west_east
Data variables: (12/21)
    WRF_GLW              (time, south_north, west_east) float32 dask.array<chunksize=(16, 336, 336), meta=np.ndarray>
    WRF_IVT_U            (time, south_north, west_east) float32 dask.array<chunksize=(16, 336, 336), meta=np.ndarray>
    WRF_IVT_V            (time, south_north, west_east) float32 dask.array<chunksize=(16, 336, 336), meta=np.ndarray>
    WRF_MSLP             (time, south_north, west_east) float32 dask.array<chunksize=(16, 336, 336), meta=np.ndarray>
    WRF_OLR              (time, south_north, west_east) float32 dask.array<chunksize=(16, 336, 336), meta=np.ndarray>
    WRF_P                (time, level, south_north, west_east) float32 dask.array<chunksize=(16, 15, 336, 336), meta=np.ndarray>
    ...                   ...
    WRF_U                (time, level, south_north, west_east) float32 dask.array<chunksize=(16, 15, 336, 336), meta=np.ndarray>
    WRF_U10              (time, south_north, west_east) float32 dask.array<chunksize=(16, 336, 336), meta=np.ndarray>
    WRF_V                (time, level, south_north, west_east) float32 dask.array<chunksize=(16, 15, 336, 336), meta=np.ndarray>
    WRF_V10              (time, south_north, west_east) float32 dask.array<chunksize=(16, 336, 336), meta=np.ndarray>
    WRF_precip           (time, south_north, west_east) float32 dask.array<chunksize=(16, 336, 336), meta=np.ndarray>
    WRF_radar_composite  (time, south_north, west_east) float32 dask.array<chunksize=(16, 336, 336), meta=np.ndarray>
Attributes:
    Conventions:  CF-1.11

In [20]:
# ds_C404 = xr.open_zarr('/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/C404/C404_GP_2020.zarr')

# ds_C404 = ds_C404.drop_vars(
#     [
#         'WRF_precip_025', 'WRF_radar_composite_025', 
#         'WRF_PWAT_05', 'WRF_Q_tot_05'
#     ]
# )

# ds_C404 = ds_C404.rename({'bottom_top': 'level', 'south_north': 'latitude', 'west_east': 'longitude'})

# ds_C404['level'] = [0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 21, 24, 30, 36, 42]
# ds_C404 = ds_C404.drop_vars(['longitude', 'latitude'])
# ds_C404 = ds_C404.assign_coords(
#     XLAT=(('latitude', 'longitude'), XLAT),
#     XLONG=(('latitude', 'longitude'), XLONG)
# )
# ds_C404.attrs['Conventions'] = 'CF-1.11'

# ds_C404['time'] = xr.decode_cf(ds_C404[['time']]).time

# # Replace indexing dims with coordinate dims
# if 'XLAT' in ds_C404 and 'XLONG' in ds_C404:
#     ds_C404 = ds_C404.set_coords(['XLAT', 'XLONG'])
#     ds_C404['XLAT'] = ds_C404['XLAT'].rename({'latitude': 'south_north', 'longitude': 'west_east'})
#     ds_C404['XLONG'] = ds_C404['XLONG'].rename({'latitude': 'south_north', 'longitude': 'west_east'})
    

# # Set CF-compliant attributes for lat/lon
# ds_C404['XLAT'].attrs = {
#     "standard_name": "latitude",
#     "units": "degrees_north",
#     "long_name": "latitude"
# }
# ds_C404['XLONG'].attrs = {
#     "standard_name": "longitude",
#     "units": "degrees_east",
#     "long_name": "longitude"
# }

# # Replace ('latitude', 'longitude') with ('south_north', 'west_east')
# for var in ds_C404.data_vars:
#     dims = ds_C404[var].dims
#     # 3D fields
#     if ('latitude' in dims or 'longitude' in dims) and 'level' in dims:
#         data = ds_C404[var].data
#         ds_C404 = ds_C404.drop_vars(var)
#         ds_C404[var] = (('time', 'level', 'south_north', 'west_east'), data)
#     # 2D fields
#     elif 'latitude' in dims or 'longitude' in dims:
#         data = ds_C404[var].data
#         ds_C404 = ds_C404.drop_vars(var)
#         ds_C404[var] = (('time', 'south_north', 'west_east'), data)


In [ ]:
# target_name = '/glade/derecho/scratch/ksha/CONUS_GP_20250520_new/target_new_2020010100Z.nc'

# time_encoding = {
#     "units": "hours since 1900-01-01 00:00:00",
#     "calendar": "gregorian"
# }

# ds_C404.to_netcdf(
#     target_name,  
#     format='NETCDF4', 
#     encoding={'time': time_encoding}, 
#     mode='w'
# )